> 在线讲义：[第 12 章 · RL 后训练：Recap，告诉模型"这段做得比平时好"](https://wikieden.github.io/vla-from-scratch/lessons/12-rl-recap.html) · 本书《VLA 从零到 π0》

# 第 12 章 · RL 后训练：Recap，告诉模型"这段做得比平时好"

配套讲义：仓库里 `vla-deep-dive/notes/lessons/12-rl-recap.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `episode_return(steps, success)` | 30 步成功 −29 > 50 步成功 −49 > 失败 −149 |
| 第 2 步 | `advantage(40, after)` | +5 / 0 / −8 |
| 第 3 步 | `label(advs)` | 门槛 −4.3，7 / 10 段 positive |

数字是讲义里编的例子：每段 N = 10 步，每步奖励 −1，价值 = −（还剩几步）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

**函数 `episode_return`**：Recap 的奖励：每步 −1；最后成功得 0，失败得 −C_fail。

In [ ]:
def episode_return(steps, success, c_fail=100):
    """Recap 的奖励：每步 −1；最后成功得 0，失败得 −C_fail。"""
    return -(steps - 1) + (0 if success else -c_fail)

**函数 `advantage`**：A = N 步奖励之和 + V(后) − V(前)，V = −还剩几步。

In [ ]:
def advantage(remaining_before, remaining_after, n=10):
    """A = N 步奖励之和 + V(后) − V(前)，V = −还剩几步。"""
    return -n + (-remaining_after) - (-remaining_before)

**函数 `label`**：优势高于本任务 30% 分位数的贴 positive；人纠正的那些一律 positive。

In [ ]:
def label(advs, q=0.3, corrections=()):
    """优势高于本任务 30% 分位数的贴 positive；人纠正的那些一律 positive。"""
    thr = float(np.quantile(advs, q))
    return thr, [a > thr or i in corrections for i, a in enumerate(advs)]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 奖励：越快成功越好")
for steps, ok in ((30, True), (50, True), (50, False)):
    print(f"  {steps} 步、{'成功' if ok else '失败'}：总奖励 {episode_return(steps, ok)}")

In [ ]:
print("\n第二步 优势：这一段比平时好多少")
for name, before, after in (("顺利", 40, 25), ("一般", 40, 30), ("衣服掉了", 40, 38)):
    print(f"  {name}：还剩 {before} → {after}，A = {advantage(before, after):+d}")

In [ ]:
print("\n第三步 按分位数贴标签")
advs = [5, 0, -8, 3, -4, 2, -6, 1, -5, 4]  # 一个任务里 10 段的优势（编的）
thr, tags = label(advs)
print(f"  30% 分位数门槛 {thr:+.1f}；positive {sum(tags)} / {len(tags)} 段")

## 3. 代码里才看得到的

1. 最后一步成功得 0 而不是 −1，所以总奖励是 −(T − 1)，比讲义说的"−用了多少步"差 1，不影响"越快越好"。
2. `label` 用 NumPy 默认的线性插值分位数；10 个数的 30% 分位数落在第 3、4 小的数之间，所以是 −4.3 而不是某个原始值。
3. `C_fail` 默认取 100，只要比任何一局的步数都大，失败就一定比成功差。

## 4. 自己改一改

- 把 `C_fail` 改成 10，看 50 步失败和 30 步成功谁的总奖励高。
- 给 `label` 加上人纠正的片段编号，看 positive 比例变成多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 看老师只能学到老师那样，自己练才能更好。
> 价值是导航的"还要多久"，优势是"比预计快了多少"。
> 好段贴正，差段贴负，上场只说"正"。
> 练、评、学，转几圈。